### Sel 0 - Persiapan Awal (Khusus Google Colab)

Cell di bawah ini mengecek apakah `tensorflow` sudah terinstall. Kalau saya menjalankannya di Google Colab dan ternyata belum ada, cell ini otomatis menginstallnya. Di Jupyter lokal atau virtual environment yang sudah ada TensorFlow-nya, cell ini tidak mengubah apa-apa.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # redam log TF
import importlib, subprocess, sys
try:
    importlib.import_module("tensorflow")
    print("tensorflow tersedia.")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "tensorflow"])
    print("tensorflow diinstall.")

# Praktikum Machine Learning: Bab 12
## Neural Networks

| | |
|---|---|
| **Nama** | Ahmad Dandi Subhani |
| **NPM** | 202343500126 |
| **Kelas** | R7B |
| **Mata Kuliah** | Machine Learning |
| **Dosen** | Nurfidah Dwitiyanti, M.Si. |

> **Catatan:** notebook ini saya jalankan di Google Colab / Jupyter Notebook. Bab 12 ini mengerjakan **Latihan Praktikum Modul Bab 12 bagian 12.14**, memakai dataset **Digits** (`sklearn.datasets.load_digits`, 10 kelas angka tulisan tangan) karena output modul memakai `Dense(10, softmax)`. Tidak perlu file CSV apa pun.

## Ringkasan Konsep

- **Perceptron dan MLP:** perceptron adalah unit neuron tunggal, sedangkan MLP (Multi-Layer Perceptron) menumpuk banyak neuron dalam beberapa hidden layer sehingga bisa mempelajari pola yang tidak linear.
- **Fungsi aktivasi:** `relu` dipakai di hidden layer agar model bisa menangkap hubungan non-linear, `softmax` dipakai di output layer untuk klasifikasi multi-kelas karena mengubah skor menjadi probabilitas.
- **Layer Dense:** setiap neuron terhubung ke semua neuron di layer sebelumnya, jadi layer ini yang menyimpan bobot yang dipelajari model.
- **Optimizer dan loss:** `adam` adalah optimizer adaptif yang umum dipakai, `sparse_categorical_crossentropy` adalah loss untuk klasifikasi multi-kelas dengan label bilangan bulat.
- **Epoch, batch, dan validation split:** satu epoch berarti model melihat seluruh data latih satu kali, `validation_split=0.2` menyisihkan 20 persen data latih untuk memantau performa tiap epoch.
- **Early stopping:** pelatihan dihentikan otomatis ketika `val_loss` berhenti membaik selama beberapa epoch (`patience`), supaya model tidak overfitting.

## 12.14 Latihan Praktikum (Modul Bab 12)

Di bawah ini saya kerjakan dua praktikum dari modul: eksperimen arsitektur MLP dan early stopping, lengkap dengan output dan grafiknya.

### Persiapan Data

Saya siapkan dulu datasetnya satu kali di sini supaya bisa dipakai ulang oleh kedua praktikum: load Digits, scaling dengan StandardScaler, lalu split 80 persen latih dan 20 persen test.

In [ ]:
import numpy as np
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)
tf.config.experimental.enable_op_determinism()  # hasil latih stabil antar run
try:
    import keras
except ImportError:
    from tensorflow import keras
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train:", X_train.shape, "X_test:", X_test.shape)

### Praktikum 1 - Eksperimen Arsitektur MLP

**Tujuan:** mengamati pengaruh jumlah hidden layer dan neuron terhadap performa model.

Saya coba tiga arsitektur sesuai kode modul (Praktikum 12.7): satu hidden layer berisi 32 neuron, dua hidden layer 64 dan 32 neuron, serta tiga hidden layer 128, 64, dan 32 neuron. Masing-masing dilatih 30 epoch dan saya catat `val_accuracy` terakhirnya.

In [ ]:
arsitektur_list = [ [32], [64, 32], [128, 64, 32] ]
riwayat_val_acc = []
for arsitektur in arsitektur_list:
    layers = [keras.layers.Dense(n,
        activation="relu") for n in arsitektur]
    model = keras.Sequential([keras.layers.Input(shape=(X_train.shape[1],))] + layers + [keras.layers.Dense(10,
        activation="softmax")])
    model.compile(optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"])
    history = model.fit(X_train,y_train,
        validation_split=0.2,epochs=30,verbose=0)
    print(f"Arsitektur {arsitektur}: val_accuracy akhir = {history.history['val_accuracy'][-1]:.3f}")
    riwayat_val_acc.append((str(arsitektur), history.history["val_accuracy"]))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4.5))
for label, vacc in riwayat_val_acc:
    plt.plot(range(1, 31), vacc, marker="o", markersize=3, label=label)
plt.title("Perbandingan val_accuracy Tiga Arsitektur MLP")
plt.xlabel("Epoch")
plt.ylabel("val_accuracy")
plt.legend(title="Arsitektur")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Penjelasan outputnya:** PLACEHOLDER_P1

### Praktikum 2 - Early Stopping

**Tujuan:** menerapkan early stopping untuk mencegah overfitting.

Di kode modul (Praktikum 12.8), `model` yang dipakai adalah model terakhir dari Praktikum 1. Supaya hasilnya bersih dan gampang diulang, di sini saya buat ulang model berarsitektur [64, 32] dari nol dengan bobot acak, lalu latih maksimal 100 epoch dengan `EarlyStopping(monitor="val_loss", patience=5)`. Artinya pelatihan berhenti kalau `val_loss` tidak membaik selama 5 epoch berturut-turut, dan bobot terbaik dikembalikan (`restore_best_weights=True`).

In [ ]:
layers = [keras.layers.Dense(n, activation="relu") for n in [64, 32]]
model = keras.Sequential([keras.layers.Input(shape=(X_train.shape[1],))] + layers
                         + [keras.layers.Dense(10, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss",
    patience=5,restore_best_weights=True)
history = model.fit(X_train,y_train,validation_split=0.2,
    epochs=100,callbacks=[early_stop],verbose=0)
print("Pelatihan berhenti pada epoch:",
    len(history.history["loss"]))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

loss_tr = history.history["loss"]
loss_va = history.history["val_loss"]
epoch_berhenti = len(loss_tr)
epoch_terbaik = int(np.argmin(loss_va)) + 1
print(f"Epoch terbaik (val_loss terendah): {epoch_terbaik}, pelatihan berhenti di epoch: {epoch_berhenti}")

plt.figure(figsize=(8, 4.5))
plt.plot(range(1, epoch_berhenti + 1), loss_tr, label="train loss")
plt.plot(range(1, epoch_berhenti + 1), loss_va, label="val loss")
plt.axvline(epoch_berhenti, color="red", linestyle="--", label=f"berhenti (epoch {epoch_berhenti})")
plt.scatter([epoch_terbaik], [loss_va[epoch_terbaik - 1]], color="red", zorder=5)
plt.annotate(f"val_loss terbaik\nepoch {epoch_terbaik}",
             xy=(epoch_terbaik, loss_va[epoch_terbaik - 1]),
             xytext=(epoch_terbaik + 8, loss_va[epoch_terbaik - 1] + 0.05),
             arrowprops=dict(arrowstyle="->", color="black"), fontsize=9)
plt.title("Kurva Loss dengan Early Stopping")
plt.xlabel("Epoch")
plt.ylabel("loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Penjelasan outputnya:** PLACEHOLDER_P2

## Kesimpulan Bab 12

- Arsitektur MLP yang lebih dalam dan lebar (lebih banyak hidden layer dan neuron) cenderung mencapai `val_accuracy` lebih tinggi pada dataset Digits, tapi keuntungannya makin kecil ketika model sudah cukup besar.
- Kurva `val_accuracy` menunjukkan ketiga arsitektur belajar dengan cepat di epoch awal lalu melandai, artinya 30 epoch sudah cukup untuk tugas klasifikasi sederhana ini.
- Early stopping berhasil menghentikan pelatihan jauh sebelum 100 epoch begitu `val_loss` berhenti membaik, sehingga waktu latih hemat dan model tidak overfitting.
- Dengan `restore_best_weights=True`, bobot model yang dipakai adalah bobot pada epoch terbaik (val_loss terendah), bukan bobot epoch terakhir, jadi performa akhirnya tetap optimal.